# Thành viên 3 – JOIN Optimization
Kết hợp tất cả bảng bằng Standard Join và Broadcast Join, so sánh hiệu năng.

In [ ]:
import os
import urllib.request

# 1. Tạo thư mục chuẩn
hadoop_bin = "C:\\hadoop\\bin"
os.makedirs(hadoop_bin, exist_ok=True)

# 2. Link tải file trực tiếp (dành cho Hadoop 3.3.0)
winutils_url = "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.0/bin/winutils.exe"
hadoop_dll_url = "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.0/bin/hadoop.dll"

# 3. Tiến hành tải
print("⏳ Đang tải winutils.exe...")
urllib.request.urlretrieve(winutils_url, os.path.join(hadoop_bin, "winutils.exe"))

print("⏳ Đang tải hadoop.dll...")
urllib.request.urlretrieve(hadoop_dll_url, os.path.join(hadoop_bin, "hadoop.dll"))

print(f"✅ HOÀN TẤT! Các file đã được cài đặt an toàn tại: {hadoop_bin}")

## 1. Load dữ liệu đã làm sạch & aggregate

In [ ]:
from feature_engineering import (
    aggregate_bureau, aggregate_bureau_balance,
    aggregate_previous_application, aggregate_installments,
    aggregate_credit_card, aggregate_pos_cash
)

DATA = '../data'
app   = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/application_train.csv')
bureau= spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/bureau.csv')
bb    = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/bureau_balance.csv')
prev  = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/previous_application.csv')
inst  = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/installments_payments.csv')
cc    = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/credit_card_balance.csv')
pos   = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/POS_CASH_balance.csv')

agg_bureau = aggregate_bureau(bureau)
agg_prev   = aggregate_previous_application(prev)
agg_inst   = aggregate_installments(inst)
agg_cc     = aggregate_credit_card(cc)
agg_pos    = aggregate_pos_cash(pos)

print('All aggregations done.')

## 2. Standard Join (không dùng Broadcast)

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')  # tắt broadcast tự động

t0 = time.time()
result_standard = app
for agg_df in [agg_bureau, agg_prev, agg_inst, agg_cc, agg_pos]:
    result_standard = result_standard.join(agg_df, on='SK_ID_CURR', how='left')

count_std = result_standard.count()
time_std = time.time() - t0
print(f'Standard Join — rows: {count_std:,}  |  time: {time_std:.1f}s')

## 3. Broadcast Join

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(50 * 1024 * 1024))

t0 = time.time()
result_broadcast = app
for agg_df in [agg_bureau, agg_prev, agg_inst, agg_cc, agg_pos]:
    result_broadcast = result_broadcast.join(broadcast(agg_df), on='SK_ID_CURR', how='left')

count_bc = result_broadcast.count()
time_bc = time.time() - t0
print(f'Broadcast Join — rows: {count_bc:,}  |  time: {time_bc:.1f}s')

## 4. So sánh hiệu năng

In [ ]:
labels = ['Standard Join', 'Broadcast Join']
times  = [time_std, time_bc]

fig, ax = plt.subplots(figsize=(6,4))
bars = ax.bar(labels, times, color=['tomato', 'mediumseagreen'])
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{t:.1f}s', ha='center', fontsize=11)
ax.set_ylabel('Time (seconds)')
ax.set_title('JOIN Performance: Standard vs Broadcast')
plt.tight_layout()
plt.savefig('../reports/EDA_visualizations/join_performance.png', dpi=150)
plt.show()

print(f'Speedup: {time_std / time_bc:.2f}x')

## 5. Xem Execution Plan

In [ ]:
print('=== Standard Join Plan ===')
result_standard.explain(mode='formatted')

print('\n=== Broadcast Join Plan ===')
result_broadcast.explain(mode='formatted')

## 6. Lưu kết quả cuối cùng

In [ ]:
result_broadcast.write \
    .mode('overwrite') \
    .option('compression', 'snappy') \
    .parquet('../output/final_features.parquet')

print('Final features saved to ../output/final_features.parquet')

In [ ]:
spark.stop()